# Lightweight ASL Word Recognition (5 Words) - Kaggle Notebook

This notebook trains a **lightweight sign-language word recognizer** for these words:
**HELLO, THANKYOU, YES, NO, SORRY**.

It uses **MediaPipe hand landmarks** + a compact sequence model, which is ideal for:
- School project simplicity
- Fast training on Kaggle
- Low-latency streaming/webcam inference after export

## Dataset Setup for Your 5 Words

This notebook is configured to train on only these labels:
- `hello`
- `thankyou`
- `yes`
- `no`
- `sorry`

Your dataset structure (like your screenshot) is supported directly, for example:
- `HELLO/HELLO_clip1.avi`
- `HELLO/HELLO_clip1_frames/` (images)

It can read from either:
1. **Video files** (`.avi`, `.mp4`, etc.)
2. **Frame folders** (clip folders containing images)

Default is to use **videos** when present (recommended for cleaner temporal learning).
Alphabet folders/classes are ignored automatically because only the 5 target words are matched.

In [ ]:
# Kaggle dependency setup (force official MediaPipe package)
!pip -q uninstall -y mediapipe mediapipe-silicon mediapipe-nightly
!pip -q install --no-cache-dir "protobuf<5" "mediapipe==0.10.14" "opencv-python-headless<4.11" scikit-learn==1.5.1

print(
    "✅ Dependencies installed. If this is a fresh install, restart kernel before running next cells."
)

In [ ]:
import os
import warnings

# Reduce verbose native logs from TensorFlow/MediaPipe
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import cv2
import json
import random
import numpy as np
import pandas as pd
from glob import glob

import mediapipe as mp
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

# Hide known noisy protobuf deprecation warning in Kaggle runtime
warnings.filterwarnings(
    "ignore",
    message=r"SymbolDatabase.GetPrototype\(\) is deprecated.*",
    category=UserWarning,
    module="google.protobuf.symbol_database",
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("MediaPipe version:", getattr(mp, "__version__", "unknown"))
print("MediaPipe path:", getattr(mp, "__file__", "<namespace>"))

if not hasattr(mp, "solutions"):
    raise RuntimeError(
        "Loaded mediapipe does not expose mp.solutions. This usually means a conflicting package/module. "
        "Re-run Cell 3, restart kernel, then re-run from Cell 4."
    )

In [ ]:
# =========================
# Configuration
# =========================
DATA_ROOT = "/kaggle/input"  # Point this to your dataset root if needed
VIDEO_EXTS = (".mp4", ".avi", ".mov", ".mkv", ".webm")
FRAME_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

# Your exact school-project vocabulary (5 words)
TARGET_WORDS = ["hello", "thankyou", "yes", "no", "sorry"]
REQUIRE_ALL_TARGET_WORDS = True

# Choose source type: "video", "frames", or "auto"
# - video: use clips only
# - frames: use frame folders only
# - auto: prefer videos if found, otherwise frames
PREFERRED_SOURCE = "video"

MAX_FRAMES = 40  # Number of timesteps per sample
MAX_SAMPLES_PER_CLASS = 250  # Safety cap for speed/memory
IMG_SIZE = (320, 240)
TEST_SIZE = 0.15
VAL_SIZE = 0.15
BATCH_SIZE = 8  # small dataset -> smaller batch gives more steps/epoch
EPOCHS = 45

CACHE_DIR = "/kaggle/working/cache_landmarks"
os.makedirs(CACHE_DIR, exist_ok=True)

print("Config ready.")

In [ ]:
import re


def normalize_token(s: str) -> str:
    s = s.strip().lower()
    s = re.sub(r"[^a-z0-9]", "", s)
    return s


TARGET_MAP = {normalize_token(w): w for w in TARGET_WORDS}


def infer_label_from_path(path: str):
    parts = os.path.normpath(path).split(os.sep)
    for p in reversed(parts):
        t = normalize_token(p)
        if t in TARGET_MAP:
            return TARGET_MAP[t]
    return None


def discover_video_files(data_root):
    files = []
    for ext in VIDEO_EXTS:
        files.extend(glob(os.path.join(data_root, "**", f"*{ext}"), recursive=True))
    files = sorted(list(set(files)))

    records = []
    for fp in files:
        label = infer_label_from_path(fp)
        if label is None:
            continue
        records.append((fp, label, "video"))

    return pd.DataFrame(records, columns=["media_path", "label", "source_type"])


def discover_frame_dirs(data_root):
    frame_dirs = []
    for root, _, files in os.walk(data_root):
        img_files = [f for f in files if f.lower().endswith(FRAME_EXTS)]
        if len(img_files) > 0:
            frame_dirs.append(root)

    frame_dirs = sorted(list(set(frame_dirs)))
    records = []
    for d in frame_dirs:
        label = infer_label_from_path(d)
        if label is None:
            continue
        records.append((d, label, "frames"))

    return pd.DataFrame(records, columns=["media_path", "label", "source_type"])


df_video = discover_video_files(DATA_ROOT)
df_frames = discover_frame_dirs(DATA_ROOT)

print("Video samples found:", len(df_video))
print("Frame folders found:", len(df_frames))

if PREFERRED_SOURCE == "video":
    df_all = df_video if len(df_video) > 0 else df_frames
elif PREFERRED_SOURCE == "frames":
    df_all = df_frames if len(df_frames) > 0 else df_video
else:
    df_all = df_video if len(df_video) > 0 else df_frames

if len(df_all) == 0:
    raise ValueError(
        "No usable samples found. Ensure your path has videos or frame folders for target words."
    )

df = df_all[df_all["label"].isin(TARGET_WORDS)].copy()

if len(df) == 0:
    raise ValueError(
        f"Found data, but none matched TARGET_WORDS={TARGET_WORDS}. Check folder names."
    )

found_labels = sorted(df["label"].unique().tolist())
missing_labels = sorted(set(TARGET_WORDS) - set(found_labels))

if REQUIRE_ALL_TARGET_WORDS and missing_labels:
    raise ValueError(f"Missing target classes in data: {missing_labels}")

# Optional cap per class for speed
df = (
    df.groupby("label", group_keys=False)
    .apply(lambda x: x.sample(min(len(x), MAX_SAMPLES_PER_CLASS), random_state=SEED))
    .reset_index(drop=True)
)

print("Using source type:", df["source_type"].iloc[0])
print("Selected labels:", sorted(df["label"].unique()))
print("Samples after cap:", len(df))
print(df["label"].value_counts())

In [ ]:
# Train/Val/Test split with robust stratification for small datasets
label_counts = df["label"].value_counts()
can_stratify = label_counts.min() >= 3

if can_stratify:
    train_df, test_df = train_test_split(
        df, test_size=TEST_SIZE, random_state=SEED, stratify=df["label"]
    )

    train_counts = train_df["label"].value_counts()
    can_stratify_val = train_counts.min() >= 2
    train_df, val_df = train_test_split(
        train_df,
        test_size=VAL_SIZE,
        random_state=SEED,
        stratify=train_df["label"] if can_stratify_val else None,
    )
else:
    print("[WARN] Very small class counts detected. Using non-stratified split.")
    train_df, test_df = train_test_split(
        df, test_size=TEST_SIZE, random_state=SEED, stratify=None
    )
    train_df, val_df = train_test_split(
        train_df, test_size=VAL_SIZE, random_state=SEED, stratify=None
    )

le = LabelEncoder()
le.fit(train_df["label"])

for split_name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(split_name, len(split_df))
    print(split_df["label"].value_counts())

class_names = list(le.classes_)
num_classes = len(class_names)
print("Classes:", class_names)
print("num_classes:", num_classes)

In [ ]:
# =========================
# MediaPipe Landmark Extraction
# Output shape per sample: [MAX_FRAMES, 126]
# 126 = 2 hands * 21 landmarks * 3 coords
# =========================
mp_hands = mp.solutions.hands
print("Using MediaPipe Hands from mp.solutions.hands")


def _empty_hand():
    return np.zeros((21, 3), dtype=np.float32)


def extract_hand_vector_from_frame(frame_bgr, hands_model):
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    result = hands_model.process(frame_rgb)

    left = _empty_hand()
    right = _empty_hand()

    if result.multi_hand_landmarks and result.multi_handedness:
        for hand_lm, handedness in zip(
            result.multi_hand_landmarks, result.multi_handedness
        ):
            pts = np.array(
                [[lm.x, lm.y, lm.z] for lm in hand_lm.landmark], dtype=np.float32
            )
            label = handedness.classification[0].label.lower()  # 'left' or 'right'
            if label == "left":
                left = pts
            else:
                right = pts

    feat = np.concatenate([left.reshape(-1), right.reshape(-1)], axis=0)
    return feat


def uniform_indices(n, max_len):
    if n <= max_len:
        return np.arange(n)
    return np.linspace(0, n - 1, max_len).astype(int)


def sort_key_natural(name):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", name)]


def pad_or_trim(seq, max_frames=MAX_FRAMES):
    seq = np.array(seq, dtype=np.float32)
    if len(seq) < max_frames:
        pad = np.zeros((max_frames - len(seq), 126), dtype=np.float32)
        seq = np.vstack([seq, pad])
    elif len(seq) > max_frames:
        seq = seq[:max_frames]
    return seq


def extract_sequence_from_video(video_path, hands_model, max_frames=MAX_FRAMES):
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total <= 0:
        frames = []
        while True:
            ok, fr = cap.read()
            if not ok:
                break
            frames.append(fr)
        cap.release()

        if len(frames) == 0:
            return np.zeros((max_frames, 126), dtype=np.float32)

        idx = uniform_indices(len(frames), max_frames)
        seq = []
        for i in idx:
            fr = cv2.resize(frames[i], IMG_SIZE)
            seq.append(extract_hand_vector_from_frame(fr, hands_model))
        return pad_or_trim(seq, max_frames)

    idx_set = set(uniform_indices(total, max_frames).tolist())
    seq = []
    frame_idx = 0
    while True:
        ok, fr = cap.read()
        if not ok:
            break
        if frame_idx in idx_set:
            fr = cv2.resize(fr, IMG_SIZE)
            seq.append(extract_hand_vector_from_frame(fr, hands_model))
        frame_idx += 1
    cap.release()
    return pad_or_trim(seq, max_frames)


def extract_sequence_from_frames_dir(frames_dir, hands_model, max_frames=MAX_FRAMES):
    frame_files = [
        os.path.join(frames_dir, f)
        for f in os.listdir(frames_dir)
        if f.lower().endswith(FRAME_EXTS)
    ]
    frame_files = sorted(
        frame_files, key=lambda x: sort_key_natural(os.path.basename(x))
    )

    if len(frame_files) == 0:
        return np.zeros((max_frames, 126), dtype=np.float32)

    idx = uniform_indices(len(frame_files), max_frames)
    seq = []
    for i in idx:
        fr = cv2.imread(frame_files[i])
        if fr is None:
            continue
        fr = cv2.resize(fr, IMG_SIZE)
        seq.append(extract_hand_vector_from_frame(fr, hands_model))

    return pad_or_trim(seq, max_frames)

In [ ]:
from tqdm.auto import tqdm


def build_xy(split_df, split_name):
    cache_x = os.path.join(CACHE_DIR, f"X_{split_name}.npy")
    cache_y = os.path.join(CACHE_DIR, f"y_{split_name}.npy")

    if os.path.exists(cache_x) and os.path.exists(cache_y):
        print(f"Loading cached {split_name} tensors...")
        X = np.load(cache_x)
        y = np.load(cache_y)
        return X, y

    X_list, y_list = [], []

    # Reuse one Hands model for the entire split (faster + fewer warnings)
    with mp_hands.Hands(
        static_image_mode=False,
        max_num_hands=2,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    ) as hands_model:
        for _, row in tqdm(
            split_df.iterrows(), total=len(split_df), desc=f"Extract {split_name}"
        ):
            media_path, label, source_type = (
                row["media_path"],
                row["label"],
                row["source_type"],
            )
            try:
                if source_type == "video":
                    seq = extract_sequence_from_video(
                        media_path, hands_model, max_frames=MAX_FRAMES
                    )
                else:
                    seq = extract_sequence_from_frames_dir(
                        media_path, hands_model, max_frames=MAX_FRAMES
                    )

                X_list.append(seq)
                y_list.append(label)
            except Exception as e:
                print(f"[WARN] Failed: {media_path} -> {e}")

    X = np.array(X_list, dtype=np.float32)
    y = le.transform(y_list).astype(np.int32)

    np.save(cache_x, X)
    np.save(cache_y, y)
    return X, y


X_train, y_train = build_xy(train_df, "train")
X_val, y_val = build_xy(val_df, "val")
X_test, y_test = build_xy(test_df, "test")

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:  ", X_val.shape, "y_val:  ", y_val.shape)
print("X_test: ", X_test.shape, "y_test: ", y_test.shape)

In [ ]:
def build_lightweight_model(timesteps=MAX_FRAMES, feat_dim=126, n_classes=5):
    inp = layers.Input(shape=(timesteps, feat_dim), name="landmarks")

    x = layers.LayerNormalization()(inp)
    x = layers.Conv1D(64, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(96, 3, padding="same", activation="relu")(x)
    x = layers.GRU(64, return_sequences=False)(x)
    x = layers.Dropout(0.30)(x)
    x = layers.Dense(64, activation="relu")(x)
    out = layers.Dense(n_classes, activation="softmax")(x)

    model = tf.keras.Model(inp, out, name="asl_word_light")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


model = build_lightweight_model(
    timesteps=MAX_FRAMES, feat_dim=126, n_classes=num_classes
)
model.summary()

In [ ]:
# Balanced class weighting + minority oversampling for tiny datasets
classes = np.unique(y_train)
class_weights_arr = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train,
)
class_weight_dict = {int(c): float(w) for c, w in zip(classes, class_weights_arr)}
print("Class weights:", class_weight_dict)


def oversample_with_landmark_jitter(X, y, noise_std=0.01):
    counts = pd.Series(y).value_counts()
    target = int(counts.max())

    X_parts = [X]
    y_parts = [y]

    for cls, cnt in counts.items():
        need = target - int(cnt)
        if need <= 0:
            continue

        idx = np.where(y == cls)[0]
        sampled_idx = np.random.choice(idx, size=need, replace=True)
        X_new = np.copy(X[sampled_idx])

        # Jitter only non-zero landmarks (keeps padded frames unchanged)
        noise = np.random.normal(0, noise_std, size=X_new.shape).astype(np.float32)
        mask = (X_new != 0).astype(np.float32)
        X_new = X_new + noise * mask

        y_new = np.full((need,), cls, dtype=y.dtype)
        X_parts.append(X_new)
        y_parts.append(y_new)

    X_bal = np.concatenate(X_parts, axis=0)
    y_bal = np.concatenate(y_parts, axis=0)

    # Shuffle
    perm = np.random.permutation(len(y_bal))
    return X_bal[perm], y_bal[perm]


X_train_bal, y_train_bal = oversample_with_landmark_jitter(
    X_train, y_train, noise_std=0.01
)
print("Train counts (original):")
print(pd.Series(y_train).value_counts().sort_index())
print("Train counts (balanced):")
print(pd.Series(y_train_bal).value_counts().sort_index())

effective_batch_size = min(BATCH_SIZE, max(4, len(X_train_bal) // 6))
print(
    "Effective batch size:",
    effective_batch_size,
    "| balanced train samples:",
    len(X_train_bal),
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=10, restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=4, min_lr=1e-5
    ),
    tf.keras.callbacks.ModelCheckpoint(
        "/kaggle/working/best_asl_word_model.keras",
        monitor="val_accuracy",
        save_best_only=True,
    ),
]

history = model.fit(
    X_train_bal,
    y_train_bal,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=effective_batch_size,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
# Evaluate
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy: {test_acc:.4f}")

y_prob = model.predict(X_test, verbose=0)
y_pred = y_prob.argmax(axis=1)

print(classification_report(y_test, y_pred, target_names=class_names))
cm = confusion_matrix(y_test, y_pred)
print("Confusion matrix:\n", cm)

In [ ]:
# Save label map + export compact TFLite model for real-time inference
label_map = {int(i): c for i, c in enumerate(class_names)}
with open("/kaggle/working/label_map.json", "w") as f:
    json.dump(label_map, f, indent=2)

# Save Keras model artifact
model.save("/kaggle/working/best_asl_word_model.keras")


def convert_to_tflite(keras_model, allow_select_tf_ops=False):
    converter = tf.lite.TFLiteConverter.from_keras_model(keras_model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]

    if allow_select_tf_ops:
        # Needed for some GRU/TensorList graphs
        converter.target_spec.supported_ops = [
            tf.lite.OpsSet.TFLITE_BUILTINS,
            tf.lite.OpsSet.SELECT_TF_OPS,
        ]
        converter._experimental_lower_tensor_list_ops = False

    return converter.convert()


tflite_path = "/kaggle/working/asl_word_light.tflite"
conversion_mode = "TFLITE_BUILTINS"

try:
    tflite_model = convert_to_tflite(model, allow_select_tf_ops=False)
except Exception as e:
    print("Built-in TFLite conversion failed. Retrying with SELECT_TF_OPS fallback...")
    print("Reason:", str(e)[:500])
    tflite_model = convert_to_tflite(model, allow_select_tf_ops=True)
    conversion_mode = "SELECT_TF_OPS_FALLBACK"

with open(tflite_path, "wb") as f:
    f.write(tflite_model)

print("Saved artifacts:")
print("- /kaggle/working/best_asl_word_model.keras")
print("- /kaggle/working/asl_word_light.tflite")
print("- /kaggle/working/label_map.json")
print("TFLite conversion mode:", conversion_mode)

if conversion_mode == "SELECT_TF_OPS_FALLBACK":
    print(
        "Note: This TFLite model uses SELECT_TF_OPS (Flex). "
        "For Python webcam inference this is fine; mobile deployment may need Flex support."
    )

In [ ]:
# Optional local webcam inference helper (run on your laptop, not Kaggle)
# This demonstrates streaming inference with the same landmark pipeline.

from collections import deque


def run_webcam_streaming_demo(keras_model, class_names, max_frames=MAX_FRAMES):
    cap = cv2.VideoCapture(0)
    buffer = deque(maxlen=max_frames)

    with mp_hands.Hands(
        static_image_mode=False,
        max_num_hands=2,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    ) as hands_model:
        while True:
            ok, frame = cap.read()
            if not ok:
                break

            frame_small = cv2.resize(frame, IMG_SIZE)
            feat = extract_hand_vector_from_frame(frame_small, hands_model)
            buffer.append(feat)

            pred_txt = "Collecting..."
            if len(buffer) == max_frames:
                x = np.array(buffer, dtype=np.float32)[None, ...]  # [1, T, 126]
                prob = keras_model.predict(x, verbose=0)[0]
                idx = int(np.argmax(prob))
                conf = float(prob[idx])
                pred_txt = f"{class_names[idx]} ({conf:.2f})"

            cv2.putText(
                frame, pred_txt, (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2
            )
            cv2.imshow("ASL Streaming Demo", frame)

            key = cv2.waitKey(1) & 0xFF
            if key == ord("q"):
                break

    cap.release()
    cv2.destroyAllWindows()


# Usage on local machine:
# run_webcam_streaming_demo(model, class_names)

## Notes for Better Accuracy

1. Keep signs consistent for the 5 words: `hello`, `thankyou`, `yes`, `no`, `sorry`.
2. Keep class counts balanced as much as possible.
3. Record from multiple people/backgrounds/lighting conditions.
4. Increase `MAX_FRAMES` to 50-60 for more dynamic words if needed (slightly slower).
5. For speed on edge devices, prefer the exported quantized TFLite model.